In [ ]:
import re
import numpy as np
import qutip as qt
import pandas as pd
import seaborn as sns
import sklearn as sk
import scipy as sp
import scipy.cluster.hierarchy as spc
import matplotlib.pyplot as plt
import matplotlib.colors as colors
from qutip import Bloch, basis
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
from datasets import load_dataset
from tqdm import tqdm
import pickle, os
import gzip
from sklearn.metrics import roc_auc_score
from scipy.stats import entropy

from functions import *

models = ['phi_4_mini', 'phi_4', 'llama4_maverick']
datasets = ['TriviaQA', 'OpenNQ']
if not os.path.exists('figures'):
    os.makedirs('figures')
if not os.path.exists('results'):
    os.makedirs('results')

# Intuition: Eigenvalues as probabilities of latent outcomes

In [ ]:
### Figure 1a
### SEMANTIC EMBEDDING ASSUMPTION ###
np.random.seed(100)
sigma = 0.5
cluster_samples = np.random.multivariate_normal([0,1,1], np.diag([sigma,sigma,sigma]), size=7)
# create outliers
cluster_samples[0] = [.5, -1, 0]
cluster_samples[1] = [.5, .5, -1]
# normalise
cluster_samples = normalise_rows(cluster_samples)

plot_bloch(cluster_samples[:7].T, file_name='3d_sphere_embs_default.png', point_size=250, figsize=(6,5))

In [ ]:
### Figure 1b
### EVs AS LATENT PROBABILITIES ###
# compute 1/n * K
gram_matrix = cluster_samples @ cluster_samples.T
gram_matrix /= gram_matrix.shape[0]
eigenvals, eigenvecs = np.linalg.eigh(gram_matrix)
print(gram_matrix.shape)
print(np.allclose(gram_matrix, eigenvecs @ np.diag(eigenvals) @ eigenvecs.T))

In [ ]:
# Figure 1 (b)
plot_eigenvals(eigenvals, file_name='prob_with_3EVs_default.png', font_size=20, figsize=(6,5))

In [ ]:
# set all EVs to zero except largest
eigenvals_0 = eigenvals.copy()
eigenvals_0[eigenvals < np.max(eigenvals)] = 0

reduced_gram_matrix = eigenvecs @ np.diag(eigenvals_0) @ eigenvecs.T
reduced_probs = np.diag(reduced_gram_matrix)
print(reduced_probs)

plot_bloch(cluster_samples[:7].T, reduced_probs, point_size=250)

In [ ]:
### GloVe example ###
## Need to download glove embeddings here
glove = load_glove_embeddings('glove.6B/glove.6B.50d.txt')

question = "Where is the Eiffel Tower?"
answers = [
    "Paris",
    "It's Paris",
    "France's Capital Paris",
    "In Paris",
    "Einstein",
    "The name is Einstein",
    "Lasagne",
]

# Compute embeddings
a_embs = np.array([sentence_embedding(ans, glove) for ans in answers])
a_embs = normalise_rows(a_embs)

In [ ]:
eigenvals, eigenvecs = spectral_decomp_gram_matrix(a_embs)

In [ ]:
plot_eigenvals(eigenvals)

In [ ]:
ranks = argsort_based_on_max_EV(eigenvals, eigenvecs)
np.array(answers)[ranks]

# Toy constant predictor example using target ground truth answers

In [ ]:
### TriviaQA with MiniLM-L6 embedder ###
### Working with marginal distribution ###
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')
dataset = load_dataset("trivia_qa", "rc.nocontext", split='validation')

In [ ]:
n_subset = 100
sampled_dataset = dataset.shuffle(seed=100).select(range(n_subset))

In [ ]:
answers = [entry['answer']['value'] for entry in sampled_dataset]
a_embs = model.encode(answers, convert_to_tensor=True).cpu().numpy()
a_embs.shape

In [ ]:
gt_matrix = a_embs.T @ a_embs / a_embs.shape[0]
risk = np.mean([cross_entropy_loss(TS_matrix(gt_matrix, temp=.5), a_emb) for a_emb in a_embs])

In [ ]:
temps = np.round([10**(i/10) for i in range(-5, 5)], 2)
B = 50 # bootstrap samples
BS_sim_results = []
for temp in tqdm(temps):
    for sample_id in range(B):
        np.random.seed(sample_id)
        sample_indices = np.random.choice(a_embs.shape[0], size=a_embs.shape[0], replace=True)
        sampled_gt = a_embs[sample_indices]
        gt_matrix = sampled_gt.T @ sampled_gt / a_embs.shape[0]
        #sampled_matrix = gt_matrix[sample_indices,:][:,sample_indices]
        TS_result = TS_matrix(gt_matrix, temp=temp)
        risk = np.mean([cross_entropy_loss(TS_result, a_emb) for a_emb in sampled_gt]).item()
        BS_sim_results += [{'temp': 1/temp, 'risk': risk, 'bootstrap_id': sample_id}]
BS_sim_results = pd.DataFrame(BS_sim_results)

In [ ]:
### Figure 2
# Constant prediction for marginal distribution of answers
font_size = 20
plt.figure(figsize=(6, 4))
sns.lineplot(BS_sim_results, x='temp', y='risk', errorbar='sd', linewidth=5)
plt.xticks(fontsize=font_size)
plt.yticks(fontsize=font_size)
plt.xlabel("Temperature", fontsize=font_size+4)
plt.ylabel("Matrix Log Risk", fontsize=font_size+4)
plt.axvline(x=1.0, color='red', linestyle='--', linewidth=2)
ax = plt.gca()
ax.set_ylim(3.45, 5.1)
ax.set_yticks(np.arange(3.4, 5, 0.5))
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
#plt.legend()
plt.savefig('figures/TS_risk_gt_dummy.png', dpi=300, bbox_inches="tight")
plt.show();

In [ ]:
gt_matrix.shape

# Evaluations with model generated answers

## Estimator convergence

In [ ]:
### EVALS WITH MODEL GENERATIONTS ###

### conditional distribution (TriviaQA) ###

# Available datasets:
# - TriviaQA
# - OpenNQ
# Available answer models:
# - phi_4
# - phi_4_mini
# - llama4_maverick

In [ ]:
triviaqa_gen = load_data('TriviaQA', 'phi_4_mini', split='validation')
print('N size: ', triviaqa_gen['question_id'].unique().shape[0])
print('--------- Data instance 0 ---------')
print(triviaqa_gen[triviaqa_gen['question_id']==0]['generated_answer'].value_counts(normalize=True))
print('--------- Data instance 1 ---------')
print(triviaqa_gen[triviaqa_gen['question_id']==1]['generated_answer'].value_counts(normalize=True))
print('--------- Data instance 5 ---------')
print(triviaqa_gen[triviaqa_gen['question_id']==5]['generated_answer'].value_counts(normalize=True))

In [ ]:
# Estimator behaviour on real-world data (TriviaQA; data instance 5)

text_answers = triviaqa_gen[triviaqa_gen['question_id']==5]['generated_answer']
emb_answers = triviaqa_gen[triviaqa_gen['question_id']==5]['embedding']
emb_answers = np.stack(emb_answers.to_numpy())

results = sim_estimator(emb_answers)

In [ ]:
# Figure 6
font_size = 20

# redline indicates max EV for all 100 answers
plot = sns.lineplot(results, x='n_size', y='max EV', errorbar='sd', linewidth=5)
sampled_embs = normalise_rows(emb_answers)
eigenvals, _ = spectral_decomp_gram_matrix(emb_answers)
est_gt = eigenvals[-1].item()
plt.xlabel('n size', fontsize=font_size+4)
plt.ylabel('Est. max Eigenvalue', fontsize=font_size+4)
plt.xticks(fontsize=font_size)
plt.yticks(fontsize=font_size)
plt.ylim(0, 0.8)
plt.axhline(y=est_gt, color='red', linewidth=3)
ax = plt.gca()
ax.set_yticks(np.arange(0, 0.9, 0.2))
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.savefig('figures/maxEV_vs_nsize_single_instance.png', dpi=300, bbox_inches="tight")
plt.show();

## Calibration and reliability diagrams

In [ ]:
rerun = True
temps = np.round([10**(i/10) for i in range(-3, 8)], 2)
if rerun:
    results = compute_TS_curves(['TriviaQA', 'OpenNQ'], models, temps)
else:
    results = pd.read_json("results/TS_results.json")

In [ ]:
### Figure 3
plot_risk_TS_curves(results, file_name='risk_entropy_vs_temp.png', font_size=11, figsize=(8, 3.5))

In [ ]:
# temperature based on the above


TS_dict = {}
if not isinstance(results, pd.DataFrame):
    results = pd.DataFrame(results)
for _, row in results.iterrows():
    dataset = row['dataset']
    model = row['model']
    optimal_temp = np.array(row['temps'])[np.argmin(row['risks'])]
    TS_dict.setdefault(dataset, {})[model] = optimal_temp



In [ ]:
TS_dict

In [ ]:
plot_all_rel_diag(
    datasets, models, bin_type='bin2cluster', TS_dict=TS_dict,
    num_bins=8, num_clusters=5
)

In [ ]:
plot_all_rel_diag(
    datasets, models, bin_type='bin2cluster',
    num_bins=8, num_clusters=5
)

In [ ]:
dataset = 'TriviaQA'
model = 'phi_4_mini'
confs, max_EVs, freq, ece_results = compute_rel_diag(dataset, model, None, bin_type='bin2cluster', num_bins=8, num_clusters=5)

In [ ]:
### Figure 1c
plot_single_rel_diag(confs, max_EVs, freq, ece_results, figsize=(2.5, 2), font_size=10, title=None, file_name='rel_diagram_bincluster_trivia_phi4mini.png')

In [ ]:
confs, max_EVs, freq, ece_results = compute_rel_diag(dataset, model, TS_dict[dataset][model], bin_type='bin2cluster', num_bins=8, num_clusters=5)

In [ ]:
### Figure 1d
plot_single_rel_diag(confs, max_EVs, freq, ece_results, figsize=(2.5, 2), font_size=10, title=None, file_name='rel_diagram_bincluster_trivia_phi4mini_TS.png')

In [ ]:
dataset = 'TriviaQA'
model = 'phi_4_mini'
confs, max_EVs, freq, ece_results = compute_rel_diag(dataset, model, None, bin_type='quantile', num_bins=8, num_clusters=1)

In [ ]:
### Figure 4a
plot_single_rel_diag(confs, max_EVs, freq, ece_results, figsize=(2.5, 2), font_size=10, title=None, file_name='rel_diagram_quantile_trivia_phi4mini.png')

In [ ]:
confs, max_EVs, freq, ece_results = compute_rel_diag(dataset, model, TS_dict[dataset][model], bin_type='quantile', num_bins=8, num_clusters=1)

In [ ]:
### Figure 4b

plot_single_rel_diag(confs, max_EVs, freq, ece_results, figsize=(2.5, 2), font_size=10, title=None, file_name='rel_diagram_quantile_trivia_phi4mini_TS.png')

In [ ]:
trivia_results = compute_multiple_rel_diag('TriviaQA', models, num_bins=8, num_clusters=5, TS_dict=None)

In [ ]:
### Figure 7a
plot_multiple_rel_diag(
    trivia_results, 'TriviaQA', models, num_bins=8, num_clusters=5,
    figsize=(7, 2.5), file_name='rel_diagram_bincluster_trivia.png'
)

In [ ]:
trivia_results = compute_multiple_rel_diag('TriviaQA', models, num_bins=8, num_clusters=5, TS_dict=TS_dict)

In [ ]:
### Figure 7b
plot_multiple_rel_diag(
    trivia_results, 'TriviaQA', models, num_bins=8, num_clusters=5,
    figsize=(7, 2.5), file_name='rel_diagram_bincluster_trivia_TS.png'
)

In [ ]:
nq_results = compute_multiple_rel_diag('OpenNQ', models, num_bins=8, num_clusters=5, TS_dict=None)

In [ ]:
### Figure 5a

plot_multiple_rel_diag(
    nq_results, 'OpenNQ', models, num_bins=8, num_clusters=5,
    figsize=(7, 2.2), file_name='rel_diagram_bincluster_nq.png'
)

In [ ]:
nq_results = compute_multiple_rel_diag('OpenNQ', models, num_bins=8, num_clusters=5, TS_dict=TS_dict)

In [ ]:
### Figure 5b

plot_multiple_rel_diag(
    nq_results, 'OpenNQ', models, num_bins=8, num_clusters=5,
    figsize=(7, 2.2), file_name='rel_diagram_bincluster_nq_TS.png'
)

In [ ]:
rerun = True
n_bootstrap = 20
if rerun:
    results = auroc_experiments(datasets, models, TS_dict, n_bootstrap=n_bootstrap)
    results.to_pickle('results/auroc_results.pkl')

In [ ]:
results = pd.read_pickle('results/auroc_results.pkl')
results = results.drop('seed', axis=1)
results = results.rename(columns={
    'ev_auroc': 'Eigenvalue', 'TS_ev_auroc': 'Eigenvalue TS', 'ent_auroc': 'Entropy', 'TS_ent_auroc': 'Entropy TS'
})
results['dataset'] = results['dataset'].replace({'TriviaQA_2k': 'TriviaQA', 'OpenNQ_2k': 'Natural Questions'})
results['model'] = results['model'].replace({'phi_4_mini': 'Phi 4 Mini', 'phi_4': 'Phi 4', 'llama4_maverick': 'Llama4 Maverick'})
results

In [ ]:
### TABLE 1 ###
table_means = results.groupby(['dataset', 'model']).mean().round(3).astype(str)
table_stds = results.groupby(['dataset', 'model']).std().div(n_bootstrap).round(3).astype(str)
table_pm = table_means + ' $\\pm$ ' + table_stds
print(table_pm.to_latex())
table_pm

In [ ]:
# Figure 8 (a)
plot_all_rel_diag_against_correctness(datasets, models, 'test', 'max_EV', 'fuzzy_correctness', bin_type='quantile', file_name='rel_diag_equal_mass_max_EV_vs_correctness.png')


In [ ]:
# Figure 8 (b)
plot_all_rel_diag_against_correctness(datasets, models, 'test', 'max_EV', 'fuzzy_correctness', bin_type='quantile',  TS_dict=TS_dict, file_name='rel_diag_equal_mass_TS_max_EV_vs_correctness.png')
